# Verified graph analysis

This notebook examines how relationship policies and query definitions change an
etymological graph. It reads one completed analysis run; it never downloads data,
constructs a full graph, or updates its input artifacts.

Generate a run with `python -m polyphasia.analysis`, then set
`POLYPHASIA_ANALYSIS_DIR` to its directory. For repeatable execution without
registering a kernel, use `scripts/execute_analysis_notebook.py --artifacts RUN_DIR
--output NEW_NOTEBOOK.ipynb` from the repository root with the notebook extra
installed. The script starts a fresh kernel using its own Python interpreter.

**Scope:** counts, paths, and rankings describe recorded assertions under explicit
modeling choices. They do not establish linguistic truth, language prevalence, or
historical influence. The bundled fixture is synthetic and tests the workflow;
its patterns were planted deliberately.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from polyphasia.analysis import validate_run

configured_directory = os.environ.get("POLYPHASIA_ANALYSIS_DIR")
if not configured_directory:
    raise RuntimeError(
        "Set POLYPHASIA_ANALYSIS_DIR to a completed analysis directory, or use "
        "scripts/execute_analysis_notebook.py --artifacts RUN_DIR --output NEW.ipynb"
    )
run_directory = Path(configured_directory).expanduser().resolve()
manifest = validate_run(run_directory)
audit = json.loads((run_directory / "audit.json").read_text(encoding="utf-8"))
analysis = json.loads((run_directory / "analysis.json").read_text(encoding="utf-8"))
if audit["schema_version"] != 1 or analysis["schema_version"] != 1:
    raise ValueError("This notebook requires audit and analysis schema version 1")
print("Artifact hashes verified against manifest.json")

## Identify the evidence

The manifest records the input hash, declared dataset identity, code revision,
environment, parameters, and execution measurements. Hash validation detects
changed run artifacts; it does not validate the source's linguistic assertions.
A dirty working tree means the Git revision alone does not identify the code:
consult the recorded implementation hashes as well.

In [ ]:
source = manifest["input"]
input_kind = source.get("kind", "unspecified")
if input_kind == "synthetic":
    display(Markdown("**Synthetic fixture run — these results are workflow checks.**"))
else:
    display(Markdown(f"**Input kind declared by this run: `{input_kind}`.**"))
provenance = {
    "input_path": source["path"],
    "input_sha256": source["sha256"],
    "input_bytes": source["bytes"],
    "dataset_version": source.get("dataset_version"),
    "source_url": source.get("source_url"),
    "description": source.get("description"),
    "git_revision": manifest["git"]["revision"],
    "working_tree_dirty": manifest["git"]["dirty"],
}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(provenance.items(), columns=["field", "value"]))
display(Markdown("**Execution parameters**"))
display(analysis["parameters"])
display(Markdown("**Recorded environment and execution measurements**"))
display(manifest["environment"])
display(manifest["execution"])

## Account for relationship selection

`root_only` retains the two forward origin/derivation relationship types used by
the historical cleaner. `normalized` also reverses their inverse relationship
types into the same direction. Exact duplicate assertions and inverse evidence
remain traceable while a projected endpoint pair has one edge with aggregated
evidence. Other relationship types are excluded from these directed projections.
Neither policy deletes cyclic nodes.

The comparison below separates input assertions, projected edges, and distinct
nodes. These quantities are not interchangeable. An edge count reduction can
reflect aggregated evidence rather than discarded evidence.

In [ ]:
projection_rows = []
for policy, projection in analysis["projections"].items():
    counts = audit["projections"][policy]
    cycles = counts["strongly_connected_components"]
    projection_rows.append({
        "policy": policy,
        "input_assertions": audit["input"]["assertions"],
        "retained_assertions": counts["retained_assertions"],
        "excluded_assertions": counts["excluded_assertions"],
        "nodes": counts["nodes"],
        "edges": counts["edges"],
        "language_seeds": projection["seeds"],
        "raw_language_nodes_excluded": projection["raw_language_nodes_excluded"],
        "self_loop_edges": counts["self_loop_edges"],
        "cyclic_nodes": cycles["cyclic_nodes"],
        "cyclic_components": cycles["cyclic_count"],
    })
display(pd.DataFrame(projection_rows))
display(pd.DataFrame([{
    "raw_assertions": audit["input"]["assertions"],
    "distinct_nodes": audit["input"]["nodes"],
    "exact_duplicate_assertions": audit["input"]["exact_duplicate_assertions"],
    "alias_normalizations": audit["input"]["alias_normalizations"],
}]))
display(Image(filename=str(run_directory / "figures" / "retention.png")))

### Check the inverse-pair assumption

Dropping inverse relationships is only lossless when their corresponding forward
facts are present. Coverage below uses unique endpoint pairs, so repeated rows
cannot inflate it. `inverse_only_pairs` identifies evidence the root-only policy
misses. Missing coverage denominators remain undefined rather than becoming zero.

In [ ]:
coverage = pd.DataFrame.from_dict(audit["inverse_coverage"], orient="index")
coverage.index.name = "forward_relationship"
display(coverage.reset_index())

## Compare query meanings on the same projections

All queries start from nodes whose exact language token matches the requested
language. An induced subgraph retains all projected edges between selected nodes.

| Query | Included nodes |
| --- | --- |
| `ancestors` | Seeds and every node that can reach a seed. |
| `descendants` | Seeds and every node reachable from a seed. |
| `root_families` | All descendants of zero-indegree roots that reach a seed; sibling branches are included. |
| `components` | Entire weakly connected components containing a seed, retaining original edge directions. |

Root-family selection can omit seeds in rootless cyclic components. The other
selections include their seeds and tolerate cycles. Seed coverage is measured
against each projection's seeds; `raw_language_nodes_excluded` separately reports
language nodes absent from that projection.

This is a sensitivity comparison, not an accuracy contest: a larger subgraph is
not automatically a better answer. Choose the query that matches the question.

In [ ]:
queries = pd.read_csv(run_directory / "queries.csv", keep_default_na=False)
display(queries)
display(Image(filename=str(run_directory / "figures" / "query-comparison.png")))

### Quantify disagreement

Jaccard overlap is intersection size divided by union size. The left-only and
right-only counts expose nodes selected by only one query. Similar counts alone
do not establish that two queries selected the same nodes. The JSON artifact also
records hashes of sorted selected node labels for comparison across runs.

In [ ]:
overlaps = pd.read_csv(run_directory / "overlaps.csv", keep_default_na=False)
display(overlaps)

## Separate word-origin and derivational rankings

Rankings are computed within each projection's **ancestor** selection. Each
relationship family is ranked separately by distinct direct neighbors:
`out_degree` counts direct targets and `in_degree` counts direct sources. Repeated
source assertions do not multiply these degrees. A high direct degree is not a
count of all descendants and does not measure historical or cultural influence.

`affix_like` is a spelling heuristic for labels beginning or ending with `-`.
It is useful for inspecting affix-shaped hubs, but is not a morphological
classification. In the synthetic fixture, those hubs were constructed to test
this distinction. Display limits and tie handling come from the analysis run.

In [ ]:
rankings = pd.read_csv(run_directory / "rankings.csv", keep_default_na=False)
with pd.option_context("display.max_rows", 100):
    display(rankings)
display(Image(filename=str(run_directory / "figures" / "rankings.png")))

## Inspect paths and their source assertions

Each trace searches for a zero-indegree root reaching the requested word within
recorded depth/node bounds. `found` means a path was found, not that it is the
unique etymology. `rootless` records a completed search without such a root;
`bounded` records an incomplete search, and `missing` means the word is absent
from that projection. A word with zero indegree can form a one-node path.

Each displayed edge retains relationship types and source assertion samples.
Assertion IDs count **parsed nonblank input records**, not physical file lines.
Use the manifest input hash to identify the source, then recover the corresponding
record. `reversed` records an inverse assertion oriented into the canonical
root-to-derivative direction. Samples may be truncated: consult each edge's
`assertion_count` and `evidence_truncated` in `analysis.json`.

In [ ]:
trace_rows = []
evidence_rows = []
for policy, projection in analysis["projections"].items():
    for trace in projection["traces"]:
        trace_rows.append({
            "policy": policy,
            "word": trace["word"],
            "status": trace["status"],
            "path": " → ".join(trace["path"]),
            "distance": trace["distance"],
            "visited_nodes": trace["visited_nodes"],
            "bounds_hit": ", ".join(trace["bounds_hit"]),
        })
        for edge in trace["edges"]:
            for evidence in edge["evidence_sample"]:
                evidence_rows.append({
                    "policy": policy,
                    "query_word": trace["word"],
                    "canonical_source": edge["source_node"],
                    "canonical_target": edge["target_node"],
                    "edge_assertion_count": edge["assertion_count"],
                    "evidence_truncated": edge["evidence_truncated"],
                    **evidence,
                })
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(trace_rows))
if evidence_rows:
    display(pd.DataFrame(evidence_rows).head(40))
    print(f"Showing {min(len(evidence_rows), 40)} of {len(evidence_rows)} sampled assertions.")
else:
    print("No edge evidence samples: inspect the trace statuses above.")

## Interpret the run before making a claim

Use the tables to answer three bounded questions:

1. How many assertions and language nodes does each relationship policy exclude?
2. How much do ancestor selection, root families, and weak components disagree,
   and how much of the seed set does each query retain?
3. Which direct-degree results change with relationship type, and which example
   paths can be checked against their source assertion IDs?

A synthetic run demonstrates the mechanics and expected counterexamples. Claims
about the historical dataset require an identified full-data run and examination
of its reported limitations. Word labels can conflate senses and historical
stages; graph roots are zero-indegree nodes in the chosen projection, not proven
linguistic origins. Recorded runtime and memory measurements describe this run
and its environment, not a controlled benchmark against an alternative.

The JSON and CSV artifacts contain the numerical evidence behind these figures.
Preserve the manifest, artifacts, and identified source together when sharing a
result. Historical exploratory notebooks are separate from this verified path.